# Profiling the EHL GPU solver

This notebook loads the profiling results produced by a single GPU run and
extracts diagnostics; 
the evidence is then used for a second repo (`cuda-ehl-demo`), 
a CUDA-optimized version of the solver to be used in the next iterations of the project.

The single profiled run corresponds to:

$$
\mathrm{St} = 1000,\quad n = 2,\quad N_r = 3000,\quad N_t = 10000.
$$

This test was run on the EPFL Kuma cluster. One caveat to acknowledge from the get-go: 

The run was profiled with CUPTI (JAX built-in profiler). Trace warnings
  suggest that some activity events were dropped -> the absolute GPU durations
  are a lower bound. The kernel identities and call counts (tables below) are however 
  deemed reliable and sufficient to pinpoint the bottlenecks.


Joaquin Garcia-Suarez (joquin.garciasuarez@epfl.ch), 2026.


In [1]:
# Load packages

import gzip
import json
from collections import Counter
from pathlib import Path

import pandas as pd
import plotly.express as px

In [2]:
# Aux
# Resolve repo root whether the notebook is launched from repo root or notebooks/
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

profiling_root = repo_root / "profiling" / "data"
summary_file = profiling_root / "baseline_summary.json"


def find_trace_file(root: Path) -> Path:
    """Find the newest Chrome trace file under profiling/data."""
    matches = list(root.rglob("*.trace.json.gz"))
    if not matches:
        raise FileNotFoundError("No Chrome trace (*.trace.json.gz) found under profiling/data") #error
    return max(matches, key=lambda p: p.stat().st_mtime) 


trace_file = find_trace_file(profiling_root)
print("Summary:", summary_file)
print("Trace:  ", trace_file)

Summary: /home/aj/Desktop/Work/gpu-ehl-rest-coeff/profiling/data/baseline_summary.json
Trace:   /home/aj/Desktop/Work/gpu-ehl-rest-coeff/profiling/data/jax_trace_baseline/plugins/profile/2026_08_16_20_52_38/kh012.trace.json.gz


In [3]:
# Test data load

with open(summary_file) as f:
    summary = json.load(f)

V0 = summary["initial_velocity"]
Vf = summary["final_velocity"]
summary["restitution_coefficient"] = abs(Vf / V0)

print(json.dumps(summary, indent=2))

{
  "case": {
    "Stokes": 1000.0,
    "exponent": 2.0,
    "NR": 3000,
    "NT": 10000
  },
  "device": "cuda:0",
  "wall_time_seconds": 95.21076852083206,
  "initial_velocity": -0.9999523162841797,
  "final_velocity": 0.9694269299507141,
  "restitution_coefficient": 0.9694731580332772
}


## Loading the Chrome trace

The JAX profiler writes a Chrome-trace-compatible JSON file (`*.trace.json.gz`).

We parse only the complete GPU execution events (`ph == "X"` on the GPU process)
and aggregate them by kernel name.


In [4]:
with gzip.open(trace_file, "rt", encoding="utf-8") as f:
    trace = json.load(f)

events = trace["traceEvents"]

# Map process IDs to names
pid_name = {}
for ev in events:
    if ev.get("ph") == "M" and ev.get("name") == "process_name":
        pid_name[ev["pid"]] = ev["args"]["name"]

print("Processes found:")
for pid, name in pid_name.items():
    print(f"  pid {pid}: {name}")

# The GPU process is the one whose name contains "GPU"
gpu_pid = next((pid for pid, name in pid_name.items() if "GPU" in name), None)
print(f"\nGPU pid: {gpu_pid}")

# Keep only GPU duration events
gpu_events = [
    ev for ev in events
    if ev.get("pid") == gpu_pid and ev.get("ph") == "X"
]
print(f"GPU duration events: {len(gpu_events):,}")

Processes found:
  pid 1: /device:GPU:0
  pid 701: /host:CPU

GPU pid: 1
GPU duration events: 265,381


In [7]:
# Aux: classifier function
def categorize_kernel(name: str) -> str:
    """Map a CUDA kernel name to a high-level category."""
    name_l = name.lower()
    if "getrf" in name_l:
        return "LU factorization"
    if "trsm" in name_l or "trsv" in name_l:
        return "Triangular solve"
    if "gemm" in name_l or "xmma" in name_l:
        return "GEMM"
    if "pivot" in name_l or "permutation" in name_l:
        return "Pivot / permutation"
    if "memcpy" in name_l or "memset" in name_l:
        return "Memory"
    return "Other"


records = []
for ev in gpu_events:
    name = ev.get("name", "<unknown>")
    records.append({
        "kernel": name,
        "duration_us": ev.get("dur", 0),
        "category": categorize_kernel(name),
    })

df = pd.DataFrame(records)

# Aggregate by kernel
agg = (
    df.groupby(["category", "kernel"])
    .agg(duration_us=("duration_us", "sum"), count=("duration_us", "size"))
    .reset_index()
)
agg["duration_s"] = agg["duration_us"] / 1e6
agg = agg.sort_values("duration_s", ascending=False)

print(f"Unique GPU kernels: {len(agg)}")
print(f"Total traced GPU time: {agg['duration_s'].sum():.2f} s")
print(f"Note: us = microseconds")

# Assign nicknames
abbrevs = {
    "LU factorization": "LU",
    "Triangular solve": "TRSV",
    "GEMM": "GEMM",
    "Pivot / permutation": "PIVOT",
    "Memory": "MEM",
    "Other": "OTHER",
}

agg = agg.sort_values(["category", "duration_s"], ascending=[True, False])
nickname_map = {}
counters = {}
for _, row in agg.iterrows():
    name = row["kernel"]
    cat = row["category"]
    counters[cat] = counters.get(cat, 0) + 1
    nickname_map[name] = f"{abbrevs[cat]}-{counters[cat]}"

agg["nickname"] = agg["kernel"].map(nickname_map)
agg["full_name"] = agg["kernel"]

agg[["category", "nickname", "duration_us", "count", "duration_s"]].head(15)

Unique GPU kernels: 70
Total traced GPU time: 9.07 s
Note: us = microseconds


,category,nickname,duration_us,count,duration_s
4,GEMM,GEMM-1,297929.240,7930,0.297929
1,GEMM,GEMM-2,145243.020,19824,0.145243
7,GEMM,GEMM-3,99253.243,991,0.099253
3,GEMM,GEMM-4,12479.212,991,0.012479
2,GEMM,GEMM-5,10364.023,991,0.010364
0,GEMM,GEMM-6,9820.885,992,0.009821
6,GEMM,GEMM-7,6012.042,991,0.006012
5,GEMM,GEMM-8,4416.758,991,0.004417
9,LU factorization,LU-1,2516707.455,15864,2.516707
8,LU factorization,LU-2,1204789.925,7928,1.204790


In [8]:
# rank 20 most recurrent
top = agg.head(20)

fig = px.bar(
    top,
    x="duration_s",
    y="nickname",
    color="category",
    orientation="h",
    title="Top GPU kernels by total duration",
    labels={"duration_s": "Total duration (s)", "nickname": "Kernel"},
    height=600,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()



In [9]:
cat_agg = agg.groupby("category").agg(
    duration_s=("duration_s", "sum"),
    count=("count", "sum"),
).reset_index()

fig = px.pie(
    cat_agg,
    values="duration_s",
    names="category",
    title="GPU time by kernel category",
)
fig.show()

In [11]:
# rank also in terms of raw # calls

top_count = agg.sort_values("count", ascending=False).head(20)

fig = px.bar(
    top_count,
    x="count",
    y="nickname",
    color="category",
    orientation="h",
    title="Top GPU kernels by number of calls",
    labels={"count": "Number of calls", "nickname": "Kernel"},
    height=600,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## Next steps: `cuda-ehl-demo`

The dominant GPU work comes from the dense linear solve inside every time step:

```python
p_new = jnp.linalg.solve(S, RHS2)
```

This single call compiles to a sequence of LU factorization (`getrf_*`),
triangular solves (`trsm_*`, `trsv_*`), pivot permutation, plus
GEMM and memory kernels. The profile above shows that these categories account for
the bulk of GPU activity during the run.

Every dynamic step performs one dense $N_r \times N_r$ solve, then the
whole simulation executes $N_t \approx 10^4$ such solves. The current JAX-native implementation
path is general-purpose and carries overhead from:

- Python/JAX translate per step.
- Kernel launch latency for many small linear-algebra operations.
- Pivot permutation and device-device memory copies.

A dedicated CUDA implementation in `cuda-ehl-demo` can therefore target this
exact bottleneck with fused kernels, better batching, and structure-aware
solvers.